In [1]:
!pip install nba_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.1/281.1 kB 1.9 MB/s eta 0:00:00


# Get LeBron's player information and seasons

In [2]:
import pandas as pd
pd.options.display.max_columns = None

from nba_api.stats.endpoints import commonplayerinfo

player_info = commonplayerinfo.CommonPlayerInfo(player_id=2544)
lebron_season_df = player_info.available_seasons.get_data_frame()
lebron_season_df['SEASON_ID'] = lebron_season_df[lebron_season_df['SEASON_ID'].str.startswith('1') | lebron_season_df['SEASON_ID'].str.startswith('2')]
lebron_season_df = lebron_season_df[lebron_season_df['SEASON_ID'].notnull()]

# Let's get all the season ids to a list
seasons = lebron_season_df['SEASON_ID'].to_list()

# Get all games played

In [3]:
from nba_api.stats.endpoints import playergamelog

all_games_df = pd.DataFrame()

for season in seasons:
    season_year = season[1:5]  # Extract the year part from SEASON_ID (e.g., '2003' from '12003')
    next_year = str(int(season_year) + 1)[-2:]  # Get the last two digits of the next year (e.g., '04')
    
    season_str = f"{season_year}-{next_year}"  # Format season as 'YYYY-YY' (e.g., '2003-04')
    
    game_log = playergamelog.PlayerGameLog(player_id=2544, season=season_str)
    game_df = game_log.get_data_frames()[0]
    all_games_df = pd.concat([all_games_df, game_df], ignore_index=True)

all_games_df

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22003,2544,0020301179,"APR 14, 2004",CLE @ NYK,W,35,8,17,0.471,0,3,0.000,1,1,1.000,0,1,1,5,3,0,5,1,17,-6,0
1,22003,2544,0020301162,"APR 12, 2004",CLE vs. MIL,W,45,9,20,0.450,1,4,0.250,8,11,0.727,1,6,7,9,2,0,1,1,27,11,0
2,22003,2544,0020301151,"APR 10, 2004",CLE vs. MIA,W,32,4,11,0.364,0,2,0.000,2,5,0.400,1,2,3,11,0,0,2,3,10,14,0
3,22003,2544,0020301137,"APR 09, 2004",CLE @ MIA,L,42,6,14,0.429,1,1,1.000,11,15,0.733,1,6,7,8,1,0,8,2,24,-16,0
4,22003,2544,0020301128,"APR 07, 2004",CLE @ MEM,L,36,4,11,0.364,0,0,0.000,6,7,0.857,0,3,3,5,3,0,2,4,14,-11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2979,22023,2544,0022300127,"NOV 01, 2023",LAL vs. LAC,W,42,13,19,0.684,4,8,0.500,5,10,0.500,0,12,12,7,1,2,4,3,35,6,1
2980,22023,2544,0022300111,"OCT 30, 2023",LAL vs. ORL,W,33,7,17,0.412,2,6,0.333,3,4,0.750,0,3,3,4,3,1,5,0,19,5,1
2981,22023,2544,0022300100,"OCT 29, 2023",LAL @ SAC,L,39,11,19,0.579,3,8,0.375,2,6,0.333,0,15,15,8,0,0,8,2,27,-5,1
2982,22023,2544,0022300076,"OCT 26, 2023",LAL vs. PHX,W,35,7,14,0.500,1,5,0.200,6,8,0.750,1,7,8,9,2,2,5,1,21,22,1


In [4]:
all_games_df.columns

Index(['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL',
       'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA',
       'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF',
       'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE'],
      dtype='object')

# Drop unnecessary columns

In [5]:
cols_to_drop = ['SEASON_ID', 'Player_ID', 'Game_ID', 'VIDEO_AVAILABLE']
all_games_df.drop(columns=cols_to_drop, inplace=True)

# Divide categorical and continuous columns

In [6]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

LE = LabelEncoder()
SS = StandardScaler()
categorical = ['GAME_DATE', 'MATCHUP', 'WL']
continuous = ['MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA',
       'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF',
       'PTS', 'PLUS_MINUS']
for col in categorical:
    all_games_df[col] = LE.fit_transform(all_games_df[col])
for col in continuous:
    all_games_df[col] = SS.fit_transform(all_games_df[col].values.reshape(-1, 1))

In [7]:
all_games_df.isnull().sum()

GAME_DATE     0
MATCHUP       0
WL            0
MIN           0
FGM           0
FGA           0
FG_PCT        0
FG3M          0
FG3A          0
FG3_PCT       0
FTM           0
FTA           0
FT_PCT        0
OREB          0
DREB          0
REB           0
AST           0
STL           0
BLK           0
TOV           0
PF            0
PTS           0
PLUS_MINUS    0
dtype: int64

# Prepare the training and validation datasets

In [8]:
from sklearn.model_selection import train_test_split

y = all_games_df['PTS']
all_games_df.drop(columns=['PTS'], inplace=True)
X = all_games_df
X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, random_state=42)

# Find the best hyper-parameters for the model

In [9]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost

# Hyper Parameter Optimization
params = {
    "learning_rate": [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30],  # Step size shrinkage to prevent overfitting
    "max_depth": [3, 4, 5, 6, 8, 10, 12, 15],  # Maximum depth of a tree
    "min_child_weight": [1, 3, 5, 7],  # Minimum sum of instance weight (hessian) needed in a child
    "gamma": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],  # Minimum loss reduction required to make a further partition
    "colsample_bytree": [0.3, 0.4, 0.5, 0.7, 0.8],  # Subsample ratio of columns when constructing each tree
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],  # Subsample ratio of the training instances
    "n_estimators": [100, 200, 300, 400, 500],  # Number of boosting rounds
    "reg_alpha": [0, 0.01, 0.05, 0.1, 1, 10],  # L1 regularization term on weights
    "reg_lambda": [0.01, 0.05, 0.1, 1, 10],  # L2 regularization term on weights
    "scale_pos_weight": [1, 1.5, 2, 2.5, 3],  # Controls the balance of positive and negative weights
    "colsample_bylevel": [0.6, 0.7, 0.8, 0.9, 1.0],  # Subsample ratio of columns for each split
}

regressor = xgboost.XGBRegressor()
random_search = RandomizedSearchCV(regressor, param_distributions=params, n_iter=10, scoring='neg_mean_squared_error', n_jobs=-1, cv=5, verbose=3)
random_search.fit(X,y)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                        'gamma': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.15,
                                                          0.2, 0.25, 0.3],
                                        'max_depth': [3, 4, 5, 6, 8, 10, 12,
                                                      15],
                                        'min_child_weight': [1, 3, 5, 7],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500],
                                        'reg_alpha': [0, 0.01, 0.05, 0.1, 1,
                                                      10],
                                        'reg_lambda': [0.01, 0.05, 0.1, 1, 10],
                                        'scale_pos_weight': [1, 1.5, 2, 2.5, 3],
                                        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]},
                   scoring='neg_mean_squared_error', verbose=3)

In [10]:
random_search.best_estimator_

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=0.9, colsample_bynode=None, colsample_bytree=0.3,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [11]:
from xgboost import XGBRegressor
import numpy as np

model = XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=0.9, colsample_bynode=None, colsample_bytree=0.8,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.2, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=15,
             max_leaves=None, min_child_weight=3, missing=np.nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, random_state=None)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=0.9, colsample_bynode=None, colsample_bytree=0.8,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.2, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=15,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [12]:
from sklearn.metrics import mean_squared_error

preds = model.predict(X_valid)
mean_squared_error(preds, y_valid)

0.005434246478267798

In [13]:
# import pandas as pd
# from nba_api.stats.endpoints import playergamelog, teamgamelog

# # Define LeBron James' player ID (you can find this using NBA API Buddy or other methods)
# lebron_player_id = '2544'

# # Get LeBron's game logs
# lebron_game_logs = playergamelog.PlayerGameLog(player_id=lebron_player_id).get_data_frames()[0]
# lebron_game_logs.head()
# # Extract game IDs and dates
# game_ids = lebron_game_logs['Game_ID'].tolist()
# game_dates = lebron_game_logs['GAME_DATE'].tolist()

# # Initialize empty lists for data
# win_loss_ratios = []
# last_5_game_averages = []

# # Iterate over each game
# for game_id, game_date in zip(game_ids, game_dates):
#     # Get team IDs for LeBron's team and the opposing team
#     team_game_logs = teamgamelog.TeamGameLog(team_id='1610612739', season=Season.default).get_data_frames()[0]
#     opposing_team_id = team_game_logs[team_game_logs['Game_ID'] == game_id]['MATCHUP'].values[0][-3:]

#     # Get win-loss ratio for LeBron's team and the opposing team
#     lebron_team_wins = team_game_logs[team_game_logs['Game_ID'] == game_id]['W'].values[0]
#     lebron_team_losses = team_game_logs[team_game_logs['Game_ID'] == game_id]['L'].values[0]
#     opposing_team_wins = team_game_logs[team_game_logs['Game_ID'] == game_id]['W'].values[0]
#     opposing_team_losses = team_game_logs[team_game_logs['Game_ID'] == game_id]['L'].values[0]

#     # Calculate win-loss ratio
#     lebron_win_loss_ratio = lebron_team_wins / (lebron_team_wins + lebron_team_losses)
#     opposing_team_win_loss_ratio = opposing_team_wins / (opposing_team_wins + opposing_team_losses)
#     win_loss_ratios.append((lebron_win_loss_ratio, opposing_team_win_loss_ratio))

#     # Get LeBron's last 5 game averages before this game
#     last_5_games = lebron_game_logs[lebron_game_logs['GAME_DATE'] < game_date].tail(5)
#     # Get LeBron's average stats for the last 5 games
#     #
#     last_5_game_averages.append((last_5_game_avg_points, last_5_game_avg_rebounds, last_5_game_avg_assists))

# # Create a DataFrame with the collected data
# columns = ['Game_ID', 'Game_Date', 'LeBron_Win_Loss_Ratio', 'Opposing_Team_Win_Loss_Ratio',
#            'Last_5_Game_Avg_Points', 'Last_5_Game_Avg_Rebounds', 'Last_5_Game_Avg_Assists']
# lebron_stats_df = pd.DataFrame({
#     'Game_ID': game_ids,
#     'Game_Date': game_dates,
#     'LeBron_Win_Loss_Ratio': [x[0] for x in win_loss_ratios],
#     'Opposing_Team_Win_Loss_Ratio': [x[1] for x in win_loss_ratios],
#     'Last_5_Game_Avg_Points': [x[0] for x in last_5_game_averages],
#     'Last_5_Game_Avg_Rebounds': [x[1] for x in last_5_game_averages],
#     'Last_5_Game_Avg_Assists': [x[2] for x in last_5_game_averages]
# })

# # Print the DataFrame
# print(lebron_stats_df)